This notebook covers experiments with different models and evaluation of their effectiveness.
The following approaches will be checked:
- Historical Averages
    - type: baseline
    - reason: the historical average will be evaluated to provide a minimal accuracy level expected from the model
- KNN
    - type: baseline
    - reason: based on the EDA results, no linearity between the models was revealed, so knn will be evaluated as an alternative approach to the historical averages, as it essentially finds the closest data points and computes their averages.
- Poisson Regression:
    - type: baseline
    - reason: this model type perfectly fits the regression problem because of the sold_quantity as a countable value.
- CatBoost - 2 stages: Classification and Regression
    - type: advanced
    - reason: because of the zero-inflated dataset, a classification approach to predict if there will be a sale potentially (yes/no) will help to a regression model to better understand where 0 value is expected from the beginning.

In [1]:
import pandas as pd
from sklearn.metrics import mean_absolute_error
import numpy as np
pd.set_option('display.max_rows', None)

In [2]:
df = pd.read_csv("../interim_files/notebooks/complete_data_with_features.csv", parse_dates=['date'])
df.head(1)

,Unnamed: 0,flight_key,flight_number,origin,destination,date,month_name,year,weekday_name,is_weekend,...,price_bin,hist_avg_l1,hist_count_l1,hist_avg_l2,hist_count_l2,hist_avg_l3,hist_count_l3,hist_avg,hist_level_used,fold_num
0,57650,748a8fbee6f7f89f1a660434f4fd23f9,AB133,city_001,city_002,2025-11-15,November,2025.0,Saturday,True,...,High,0.571429,14.0,0.571429,14.0,0.406897,145,0.571429,1.0,1


## Train / test df preparation
The last 2 weeks of DF are reserved to the test set.

In [3]:
test_cutoff = df['date'].max() - pd.Timedelta(weeks=2)
print(f"Train / test split by date: {test_cutoff}")
train_df = df[df['date'] <= test_cutoff]
test_df = df[df['date'] > test_cutoff]
print(f"Train DF size: {train_df.shape}")
print(f"Test DF size: {test_df.shape}")

Train / test split by date: 2026-02-14 00:00:00
Train DF size: (46040, 32)
Test DF size: (6790, 32)


Model's accuracy will be evaluated  by 2 different approaches:
- technical:
    - MAE - mean absolute error
    - RMSE - root mean square error
- business - by comparing the predicted value vs the factual ones:
    - accurate_predictions: when an actual value equals to the predicted one
    - potential_waste: when an actual value is lower than the predicted one
    - potential_lost_sale: when an actual value is higher than the predicted one

In [4]:
# technical metrics evaluation
def evaluate_technical_metrics(y_true, y_predicted):

    mae = mean_absolute_error(y_true, y_predicted)
    rmse = np.sqrt((y_true - y_predicted) ** 2).mean()

    print(f"MAE:  {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")

In [5]:
# business metrics evaluation
def evaluate_business_metrics(y_true, y_predicted):
    diff = y_true - y_predicted

    results = pd.DataFrame({
        "category": ["accurate_prediction", "potential_waste", "potential_lost_sale"],
        "number of records": [
            (diff == 0).sum(),
            (diff < 0).sum(),  # predicted > actual = waste
            (diff > 0).sum()   # predicted < actual = lost sale
        ],
        "share": [
            round((diff == 0).mean() * 100, 1),
            round((diff < 0).mean() * 100, 1),
            round((diff > 0).mean() * 100, 1)
        ],
        "standard deviation": [
            0,
            diff[diff < 0].mean(),  # average over-prediction
            diff[diff > 0].mean()   # average under-prediction
        ]
    })
    print(results.to_string(index=False))

## Historical averages

The historical averages are computed based on a hierarchical approach depending on the number of available records for a given level:
- Level 1: item_id - route - day_period
- Level 2: item_id - route - is_night
- Level 3: item_id - day_period

There should be at least 10 data points available in a group to compute an average for a given combination.

In [6]:
test_df = test_df.copy()
test_df['pred_baseline'] = test_df['hist_avg'].round().clip(lower=0).astype(int)

Checking the baseline model's accuracy

In [7]:
y_true = test_df["sold_quantity"]
y_predicted = test_df["pred_baseline"]
evaluate_technical_metrics(y_true, y_predicted)
evaluate_business_metrics(y_true, y_predicted)

MAE:  0.476
RMSE: 0.476
           category  number of records  share  standard deviation
accurate_prediction               4281   63.0            0.000000
    potential_waste               1441   21.2           -1.294934
potential_lost_sale               1068   15.7            1.277154


## KNN - Key Nearest Neighbours

The main reason to try KNN is to narrow the specificity for the data points. Compared to the averages by hierarchical grouping, KNN allows to focus on the nearest data points (3, 5, 7) will be tried out, and then it computes the average. The main difference in the approach will be changing dynamically the threshold, so that the model can be adapted based on the business needs: to prevent wastage values or to minimize the lost sales.

The following features will be put into the model: item_id, hist_avg, route, pax_bin, day_period

The modelling process will include the following steps:
1. Categorical features preparation: ordinal, target encoding for the categorical features
2. Numerical and encoded categorical features scaling (KNN is sensitive to the scale)
3. Building the model with different hyperparameters: n_neighbors = 3, 5, 7
4. Trying out different thresholds

In [8]:
# categorical features processing
from category_encoders import TargetEncoder
from sklearn.preprocessing import OrdinalEncoder

pax_order = ['> 100', "100 - 150", "150 - 180", '180 <']
oe = OrdinalEncoder(categories=[pax_order])
train_df["pax_bin_enc"] = oe.fit_transform(train_df[["pax_bin"]])
test_df["pax_bin_enc"] = oe.transform(test_df[["pax_bin"]])

# target encoding
te = TargetEncoder(cols=["item_id", "route", "day_period"])
train_df[["item_id_enc", "route_enc", "day_period_enc"]] = te.fit_transform(train_df[["item_id", "route", "day_period"]], train_df["sold_quantity"])
test_df[["item_id_enc", "route_enc", "day_period_enc"]] = te.transform(test_df[["item_id", "route", "day_period"]], test_df["sold_quantity"])

In [9]:
# numerical features scaling
from sklearn.preprocessing import StandardScaler

features = ["hist_avg", "pax_bin_enc", "item_id_enc", "route_enc", "day_period_enc"]

scaler = StandardScaler()
X_train = scaler.fit_transform(train_df[features])
X_test = scaler.transform(test_df[features])

y_train = train_df["sold_quantity"]
y_test = test_df["sold_quantity"]

In [10]:
# training and predicting
from sklearn.neighbors import KNeighborsRegressor

results ={}

for k in [3, 5, 7]:
    knn = KNeighborsRegressor(n_neighbors=k, metric="euclidean")
    knn.fit(X_train, y_train)
    results[k] = knn.predict(X_test)

In [11]:
# tuning threshold
import numpy as np

def apply_threshold(raw_pred, threshold=0.5):
    floored = np.floor(raw_pred)
    remainder = raw_pred - floored
    return (floored + (remainder >= threshold).astype(int)).astype(int).clip(min=0)

for k, raw_pred in results.items():
    for t in [0.3, 0.5, 0.7]:
        pred = apply_threshold(raw_pred, threshold=t)
        mae = mean_absolute_error(y_test, pred)
        print(f"\nk={k}, threshold={t:.1f} → MAE={mae:.3f}\n")
        evaluate_business_metrics(y_test, pred)


k=3, threshold=0.3 → MAE=0.707

           category  number of records  share  standard deviation
accurate_prediction               2996   44.1            0.000000
    potential_waste               3054   45.0           -1.257367
potential_lost_sale                740   10.9            1.294595

k=3, threshold=0.5 → MAE=0.525

           category  number of records  share  standard deviation
accurate_prediction               4115   60.6            0.000000
    potential_waste               1525   22.5           -1.356066
potential_lost_sale               1150   16.9            1.304348

k=3, threshold=0.7 → MAE=0.457

           category  number of records  share  standard deviation
accurate_prediction               4544   66.9            0.000000
    potential_waste                815   12.0           -1.460123
potential_lost_sale               1431   21.1            1.335430

k=5, threshold=0.3 → MAE=0.591

           category  number of records  share  standard deviation
accurate_p

## CatBoost Classification + Regression


In [12]:
# feature preparation
cat_features = ["item_id", "route", "day_period", "pax_bin"]
num_features = ["hist_avg"]
features = cat_features + num_features

In [13]:
# classification + regression
train_df["target_cls"] = (train_df["sold_quantity"] > 0).astype(int)
X_train = train_df[features]
X_test = test_df[features]

y_train_cls = train_df["target_cls"]
y_train_reg = train_df[train_df["target_cls"] == 1]["sold_quantity"]
X_train_reg = train_df[train_df["target_cls"] == 1][features]

In [14]:
# parameters tuning
from catboost import CatBoostClassifier, CatBoostRegressor, Pool
from sklearn.model_selection import ParameterGrid

param_grid = {
    'iterations': [300, 500, 700],
    'learning_rate': [0.03, 0.05, 0.1],
    'depth': [4, 6, 8],
}

In [15]:
val_cutoff = train_df['date'].max() - pd.Timedelta(weeks=2)
tune_train = train_df[train_df['date'] <= val_cutoff]
tune_val   = train_df[train_df['date'] > val_cutoff]

tune_train_cls = (tune_train['sold_quantity'] > 0).astype(int)
tune_val_cls   = (tune_val['sold_quantity'] > 0).astype(int)

cls_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features,
    eval_metric='F1',
    random_seed=42,
    verbose=100
)

cls_model.fit(X_train, y_train_cls)

0:	learn: 0.6041405	total: 81.7ms	remaining: 40.8s
100:	learn: 0.5920587	total: 1.46s	remaining: 5.75s
200:	learn: 0.5980164	total: 2.73s	remaining: 4.05s
300:	learn: 0.6022327	total: 4.11s	remaining: 2.72s
400:	learn: 0.6057363	total: 5.31s	remaining: 1.31s
499:	learn: 0.6086362	total: 6.61s	remaining: 0us


CatBoostClassifier(cat_features=['item_id', 'route', 'day_period', 'pax_bin'], depth=6, eval_metric='F1', iterations=500, learning_rate=0.05, random_seed=42, verbose=100)

In [16]:
# regressor
reg_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    cat_features=cat_features,
    eval_metric='MAE',
    random_seed=42,
    verbose=100
)
reg_model.fit(X_train_reg, y_train_reg)

0:	learn: 0.9862634	total: 6.13ms	remaining: 3.06s
100:	learn: 0.6623954	total: 336ms	remaining: 1.33s
200:	learn: 0.6493914	total: 607ms	remaining: 903ms
300:	learn: 0.6431816	total: 934ms	remaining: 617ms
400:	learn: 0.6384924	total: 1.23s	remaining: 305ms
499:	learn: 0.6345736	total: 1.54s	remaining: 0us


CatBoostRegressor(cat_features=['item_id', 'route', 'day_period', 'pax_bin'], depth=6, eval_metric='MAE', iterations=500, learning_rate=0.05, loss_function='RMSE', random_seed=42, verbose=100)

## Feature Importance

Analyzing which features contribute most to model predictions helps understand what drives the model's decisions.

In [17]:
# Feature importance analysis
cls_importance = cls_model.get_feature_importance()
reg_importance = reg_model.get_feature_importance()

importance_df = pd.DataFrame({
    'feature': features,
    'classifier': cls_importance,
    'regressor': reg_importance
})

importance_df = importance_df.sort_values('classifier', ascending=False)

print("CLASSIFIER Feature Importance (Zero vs Non-Zero Detection):")
print("-" * 60)
for _, row in importance_df.iterrows():
    print(f"  {row['feature']:<20} {row['classifier']:>8.2f}")

print("\n" + "=" * 60)

importance_df = importance_df.sort_values('regressor', ascending=False)

print("REGRESSOR Feature Importance (Quantity Prediction):")
print("-" * 60)
for _, row in importance_df.iterrows():
    print(f"  {row['feature']:<20} {row['regressor']:>8.2f}")

CLASSIFIER Feature Importance (Zero vs Non-Zero Detection):
------------------------------------------------------------
  hist_avg                47.00
  route                   20.09
  item_id                 13.01
  day_period              12.03
  pax_bin                  7.88

REGRESSOR Feature Importance (Quantity Prediction):
------------------------------------------------------------
  hist_avg                62.94
  route                   21.47
  item_id                  6.93
  pax_bin                  4.78
  day_period               3.89


In [18]:
# predictions with threshold
cls_proba = cls_model.predict_proba(X_test)[:, 1]
threshold = 0.5
cls_pred = (cls_proba >= threshold).astype(int)

reg_pred = reg_model.predict(X_test)
reg_pred = reg_pred.round().clip(0).astype(int)

final_pred = np.where(cls_pred == 0, 0, reg_pred)

In [19]:
# checking different thresholds
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    cls_pred = (cls_proba >= t).astype(int)
    final_pred = np.where(cls_pred == 0, 0, reg_pred)

    mae = mean_absolute_error(y_test, final_pred)
    print(f"\nthreshold={t:.1f} → MAE={mae:.2f}\n")
    evaluate_business_metrics(y_test, final_pred)


threshold=0.3 → MAE=0.59

           category  number of records  share  standard deviation
accurate_prediction               3812   56.1            0.000000
    potential_waste               2260   33.3           -1.379204
potential_lost_sale                718   10.6            1.243733

threshold=0.4 → MAE=0.53

           category  number of records  share  standard deviation
accurate_prediction               4273   62.9            0.000000
    potential_waste               1539   22.7           -1.548408
potential_lost_sale                978   14.4            1.223926

threshold=0.5 → MAE=0.50

           category  number of records  share  standard deviation
accurate_prediction               4489   66.1            0.000000
    potential_waste               1150   16.9           -1.683478
potential_lost_sale               1151   17.0            1.240660

threshold=0.6 → MAE=0.47

           category  number of records  share  standard deviation
accurate_prediction               

In [20]:

# Get CatBoost predictions (threshold=0.7, best performance)
cls_proba_best = cls_model.predict_proba(X_test)[:, 1]
cls_pred_best = (cls_proba_best >= 0.7).astype(int)
reg_pred_best = reg_model.predict(X_test).round().clip(0).astype(int)
catboost_pred = np.where(cls_pred_best == 0, 0, reg_pred_best)

y_true = test_df["sold_quantity"].values
y_true_binary = (y_true > 0).astype(int)

# Dataset composition
zero_mask = (y_true == 0)
nonzero_mask = ~zero_mask

print(f"Dataset Composition:")
print(f"  Zeros: {zero_mask.sum():,} ({zero_mask.sum()/len(y_true)*100:.1f}%)")
print(f"  Non-zeros: {nonzero_mask.sum():,} ({nonzero_mask.sum()/len(y_true)*100:.1f}%)")

print("STAGE 1: CLASSIFICATION PERFORMANCE (Zero vs Non-Zero Detection)")

from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

# Confusion matrix
cm = confusion_matrix(y_true_binary, cls_pred_best)
tn, fp, fn, tp = cm.ravel()

print(f"\nConfusion Matrix:")
print(f"                    Predicted: Zero  |  Predicted: Non-Zero")
print(f"  Actual: Zero      {tn:>6,} (TN)    |  {fp:>6,} (FP)")
print(f"  Actual: Non-Zero  {fn:>6,} (FN)    |  {tp:>6,} (TP)")

print(f"\nClassification Metrics:")
print(f"  True Negatives (correctly predicted zeros):  {tn:,} / {zero_mask.sum():,} = {tn/zero_mask.sum()*100:.1f}%")
print(f"  True Positives (correctly detected sales):   {tp:,} / {nonzero_mask.sum():,} = {tp/nonzero_mask.sum()*100:.1f}%")
print(f"  False Positives (predicted sale, was zero):  {fp:,} ({fp/zero_mask.sum()*100:.1f}% of zeros)")
print(f"  False Negatives (missed sales):              {fn:,} ({fn/nonzero_mask.sum()*100:.1f}% of non-zeros)")

print(f"\nOverall Classification Performance:")
accuracy = (tn + tp) / len(y_true)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"  Accuracy:  {accuracy*100:.1f}%")
print(f"  Precision: {precision*100:.1f}% (when model predicts sale, it's correct)")
print(f"  Recall:    {recall*100:.1f}% (% of actual sales detected)")
print(f"  F1-Score:  {f1:.3f}")
print(f"  ROC-AUC:   {roc_auc_score(y_true_binary, cls_proba_best):.3f}")

print("STAGE 2: REGRESSION PERFORMANCE (Quantity Prediction on Detected Sales)")


# Filter to records where classifier predicted non-zero
predicted_sales_mask = (cls_pred_best == 1)
actual_nonzero_predicted_sale = nonzero_mask & predicted_sales_mask

if actual_nonzero_predicted_sale.sum() > 0:
    print(f"\nRecords where model predicted a sale: {predicted_sales_mask.sum():,}")
    print(f"  Of these, actual non-zeros: {actual_nonzero_predicted_sale.sum():,} ({actual_nonzero_predicted_sale.sum()/predicted_sales_mask.sum()*100:.1f}%)")
    
    # Regression performance on correctly identified non-zeros
    reg_mae = mean_absolute_error(y_true[actual_nonzero_predicted_sale], 
                                    reg_pred_best[actual_nonzero_predicted_sale])
    exact_match = (y_true[actual_nonzero_predicted_sale] == reg_pred_best[actual_nonzero_predicted_sale]).sum()
    within_1 = (np.abs(y_true[actual_nonzero_predicted_sale] - reg_pred_best[actual_nonzero_predicted_sale]) <= 1).sum()
    within_2 = (np.abs(y_true[actual_nonzero_predicted_sale] - reg_pred_best[actual_nonzero_predicted_sale]) <= 2).sum()
    
    print(f"\nRegression Accuracy (on correctly detected sales):")
    print(f"  Exact match:   {exact_match:,} ({exact_match/actual_nonzero_predicted_sale.sum()*100:.1f}%)")
    print(f"  Within ±1 unit: {within_1:,} ({within_1/actual_nonzero_predicted_sale.sum()*100:.1f}%)")
    print(f"  Within ±2 units: {within_2:,} ({within_2/actual_nonzero_predicted_sale.sum()*100:.1f}%)")
    print(f"  MAE: {reg_mae:.3f} units")

print("\n" + "="*80)
print("OVERALL MODEL PERFORMANCE")
print("="*80)

print(f"\nFinal Predictions (after both stages):")
overall_exact = (catboost_pred == y_true).sum()
overall_mae = mean_absolute_error(y_true, catboost_pred)

# Break down by zero/non-zero
zero_correct = (catboost_pred[zero_mask] == 0).sum()
nonzero_exact = (catboost_pred[nonzero_mask] == y_true[nonzero_mask]).sum()
nonzero_within_1 = (np.abs(catboost_pred[nonzero_mask] - y_true[nonzero_mask]) <= 1).sum()
nonzero_within_2 = (np.abs(catboost_pred[nonzero_mask] - y_true[nonzero_mask]) <= 2).sum()

print(f"\nPerformance on ZEROS ({zero_mask.sum():,} records):")
print(f"  Correctly predicted zero: {zero_correct:,} ({zero_correct/zero_mask.sum()*100:.1f}%)")

print(f"\nPerformance on NON-ZEROS ({nonzero_mask.sum():,} records):")
print(f"  Exact match:        {nonzero_exact:,} ({nonzero_exact/nonzero_mask.sum()*100:.1f}%)")
print(f"  Within ±1 unit:     {nonzero_within_1:,} ({nonzero_within_1/nonzero_mask.sum()*100:.1f}%)")
print(f"  Within ±2 units:    {nonzero_within_2:,} ({nonzero_within_2/nonzero_mask.sum()*100:.1f}%)")
print(f"  MAE: {mean_absolute_error(y_true[nonzero_mask], catboost_pred[nonzero_mask]):.3f}")

print(f"\nOverall:")
print(f"  Exact match accuracy: {overall_exact:,}/{len(y_true):,} = {overall_exact/len(y_true)*100:.1f}%")
print(f"  MAE: {overall_mae:.3f}")

Dataset Composition:
  Zeros: 4,850 (71.4%)
  Non-zeros: 1,940 (28.6%)
STAGE 1: CLASSIFICATION PERFORMANCE (Zero vs Non-Zero Detection)

Confusion Matrix:
                    Predicted: Zero  |  Predicted: Non-Zero
  Actual: Zero       4,632 (TN)    |     218 (FP)
  Actual: Non-Zero   1,457 (FN)    |     483 (TP)

Classification Metrics:
  True Negatives (correctly predicted zeros):  4,632 / 4,850 = 95.5%
  True Positives (correctly detected sales):   483 / 1,940 = 24.9%
  False Positives (predicted sale, was zero):  218 (4.5% of zeros)
  False Negatives (missed sales):              1,457 (75.1% of non-zeros)

Overall Classification Performance:
  Accuracy:  75.3%
  Precision: 68.9% (when model predicts sale, it's correct)
  Recall:    24.9% (% of actual sales detected)
  F1-Score:  0.366
  ROC-AUC:   0.774
STAGE 2: REGRESSION PERFORMANCE (Quantity Prediction on Detected Sales)

Records where model predicted a sale: 701
  Of these, actual non-zeros: 483 (68.9%)

Regression Accuracy (on